# ChatGPT code


In [1]:
import numpy as np

X = np.array([
    [90, 70, 6000],
    [80, 90, 5000],
    [95, 80, 7000]
], dtype=float)

# 每一列平方和开根号
denominator = np.sqrt((X ** 2).sum(axis=0))

Z = X / denominator

print(Z)

[[0.58678323 0.50257071 0.57207755]
 [0.52158509 0.64616234 0.47673129]
 [0.6193823  0.57436653 0.66742381]]


In [2]:
denominator = np.sqrt((X ** 2).sum(axis=0))

Z = X / denominator

print("标准化矩阵：")
print(Z)

标准化矩阵：
[[0.58678323 0.50257071 0.57207755]
 [0.52158509 0.64616234 0.47673129]
 [0.6193823  0.57436653 0.66742381]]


In [3]:
weights = np.array([0.5, 0.3, 0.2])

V = Z * weights

print("加权标准化矩阵：")
print(V)

加权标准化矩阵：
[[0.29339161 0.15077121 0.11441551]
 [0.26079255 0.1938487  0.09534626]
 [0.30969115 0.17230996 0.13348476]]


In [4]:
benefit = [True, True, False]

ideal_best = np.zeros(3)
ideal_worst = np.zeros(3)

for j in range(3):

    if benefit[j]:

        ideal_best[j] = V[:, j].max()
        ideal_worst[j] = V[:, j].min()

    else:

        ideal_best[j] = V[:, j].min()
        ideal_worst[j] = V[:, j].max()

In [6]:
D_positive = np.sqrt(
    ((V - ideal_best) ** 2).sum(axis=1)
)

D_negative = np.sqrt(
    ((V - ideal_worst) ** 2).sum(axis=1)
)

In [7]:
C = D_negative / (
    D_positive + D_negative
)

In [8]:
for i, score in enumerate(C):
    print(i, score)

0 0.43104760314414564
1 0.5405694858298128
2 0.5495300660925732


In [9]:
import numpy as np

def topsis(X, weights, benefit):

    """
    TOPSIS 综合评价

    参数
    ----
    X:
        原始数据矩阵
        shape = (m, n)

    weights:
        指标权重
        shape = (n,)

    benefit:
        指标方向
        True  = 正向指标，越大越好
        False = 负向指标，越小越好

    返回
    ----
    score:
        TOPSIS 贴近度
    """

    X = np.array(X, dtype=float)
    weights = np.array(weights, dtype=float)

    m, n = X.shape

    # =========================
    # 1. 向量标准化
    # =========================

    denominator = np.sqrt(
        np.sum(X ** 2, axis=0)
    )

    Z = X / denominator

    # =========================
    # 2. 加权标准化
    # =========================

    V = Z * weights

    # =========================
    # 3. 正理想解
    # 4. 负理想解
    # =========================

    ideal_best = np.zeros(n)
    ideal_worst = np.zeros(n)

    for j in range(n):

        if benefit[j]:

            ideal_best[j] = V[:, j].max()
            ideal_worst[j] = V[:, j].min()

        else:

            ideal_best[j] = V[:, j].min()
            ideal_worst[j] = V[:, j].max()

    # =========================
    # 5. 计算距离
    # =========================

    D_positive = np.sqrt(
        np.sum(
            (V - ideal_best) ** 2,
            axis=1
        )
    )

    D_negative = np.sqrt(
        np.sum(
            (V - ideal_worst) ** 2,
            axis=1
        )
    )

    # =========================
    # 6. 计算贴近度
    # =========================

    C = D_negative / (
        D_positive + D_negative
    )

    return C, Z, V, ideal_best, ideal_worst


# =========================
# 示例
# =========================

X = np.array([
    [90, 70, 6000],
    [80, 90, 5000],
    [95, 80, 7000]
])

weights = np.array([
    0.5,
    0.3,
    0.2
])

benefit = [
    True,    # 性能
    True,    # 续航
    False    # 价格
]


C, Z, V, ideal_best, ideal_worst = topsis(
    X,
    weights,
    benefit
)


names = [
    "方案A",
    "方案B",
    "方案C"
]


print("TOPSIS贴近度：")

for name, score in zip(names, C):

    print(
        f"{name}: {score:.4f}"
    )


# =========================
# 排名
# =========================

ranking = np.argsort(C)[::-1]

print("\n排名：")

for rank, index in enumerate(ranking, 1):

    print(
        rank,
        names[index],
        C[index]
    )

TOPSIS贴近度：
方案A: 0.4310
方案B: 0.5406
方案C: 0.5495

排名：
1 方案C 0.5495300660925732
2 方案B 0.5405694858298128
3 方案A 0.43104760314414564


# Gemini code

In [10]:
import numpy as np
import pandas as pd

# ================= 1. 准备原始数据 =================
# 构建数据矩阵 (3行2列)
# 第1列是数学(正向)，第2列是缺勤(负向)
data = np.array([
    [85, 2],
    [90, 0],
    [75, 5]
], dtype=float)

# 假设两个指标一样重要，权重都是 0.5
weights = np.array([0.5, 0.5]) 

print("--- 原始数据 ---")
print(data)

# ================= 2. 数据标准化 (向量归一化) =================
# 创建一个跟 data 一样大小的空矩阵来存标准化后的数据
norm_data = np.zeros_like(data)

# 处理第一列 (数学，正向)
# 向量归一化：每个数 / 这一列所有数平方和的开方
math_col = data[:, 0]
norm_data[:, 0] = math_col / np.sqrt(np.sum(math_col**2))

# 处理第二列 (缺勤，负向)
# 注意：负向指标标准化时，为了把它变成“越大越好”，可以用 (最大值 - 当前值) 作为分子，
# 或者用经典的 TOPSIS 做法：依然先正常做向量归一化，但在下一步找理想解时反过来挑。
# 这里我们用对初学者最友好的“反转法”：用这一列的最大值减去当前值，再做归一化
absent_col = data[:, 1]
reversed_absent = np.max(absent_col) - absent_col # 翻转：5-2=3, 5-0=5, 5-5=0
norm_data[:, 1] = reversed_absent / np.sqrt(np.sum(reversed_absent**2))

print("\n--- 标准化后的数据 ---")
print(norm_data)

# ================= 3. 乘以权重 =================
# 用标准化后的矩阵，每一列乘以对应的权重
weighted_data = norm_data * weights

print("\n--- 加权后的矩阵 ---")
print(weighted_data)

# ================= 4. 寻找正、负理想解 =================
# 因为我们在第2步已经把所有指标都变成了“越大越好”
# 所以正理想解就是每一列的最大值，负理想解就是每一列的最小值
Z_plus = np.max(weighted_data, axis=0)  # 正理想解 (神仙同学)
Z_minus = np.min(weighted_data, axis=0) # 负理想解 (垫底同学)

print("\n--- 理想解 ---")
print(f"正理想解 Z+: {Z_plus}")
print(f"负理想解 Z-: {Z_minus}")

# ================= 5. 计算距离和最终得分 =================
# 用欧氏距离（勾股定理）计算每个同学到 Z+ 和 Z- 的距离
# np.sum(..., axis=1) 表示按行求和
D_plus = np.sqrt(np.sum((weighted_data - Z_plus)**2, axis=1))
D_minus = np.sqrt(np.sum((weighted_data - Z_minus)**2, axis=1))

# 计算相对贴近度 (最终得分)
scores = D_minus / (D_plus + D_minus)

# ================= 输出最终结果 =================
print("\n--- 最终成绩单 ---")
students = ['同学 A', '同学 B', '同学 C']
for i in range(3):
    print(f"{students[i]} - 到神仙的距离(D+): {D_plus[i]:.4f}, 到垫底的距离(D-): {D_minus[i]:.4f}, 最终得分: {scores[i]:.4f}")

--- 原始数据 ---
[[85.  2.]
 [90.  0.]
 [75.  5.]]

--- 标准化后的数据 ---
[[0.58725526 0.51449576]
 [0.62179968 0.85749293]
 [0.5181664  0.        ]]

--- 加权后的矩阵 ---
[[0.29362763 0.25724788]
 [0.31089984 0.42874646]
 [0.2590832  0.        ]]

--- 理想解 ---
正理想解 Z+: [0.31089984 0.42874646]
负理想解 Z-: [0.2590832 0.       ]

--- 最终成绩单 ---
同学 A - 到神仙的距离(D+): 0.1724, 到垫底的距离(D-): 0.2596, 最终得分: 0.6009
同学 B - 到神仙的距离(D+): 0.0000, 到垫底的距离(D-): 0.4319, 最终得分: 1.0000
同学 C - 到神仙的距离(D+): 0.4319, 到垫底的距离(D-): 0.0000, 最终得分: 0.0000
